# 07. Мультимодальний ввід: зображення, PDF, цитати, Files API

**Розділ довідника:** [`sections/07-multimodal.md`](../sections/07-multimodal.md)

**Потрібно:** нічого — жодних ключів, жодного GPU.
Усі розрахунки тут — чиста арифметика за формулами з документації.

**Що ви зробите:**

1. Реалізуєте **формулу візуальних токенів** і звірите її з таблицею документації.
2. Порахуєте реальну ціну зображень і PDF — і побачите, чому «розмір файлу» не визначає ціну.
3. Відтворите пастку **багато-зображувальних запитів**: раптове посилення лімітів після 20 зображень.
4. Розберете структуру цитат і різницю індексації (символи з 0, сторінки з 1).
5. Відтворите **вразливість Files API**, коли `file_id` приймається від клієнта.
6. Побудуєте вирішувач: коли пряма передача, а коли попередня обробка.

> Ключове усвідомлення: **модель не бачить пікселів.** Вона бачить патчі 28×28, і саме вони
> визначають ціну.

## Налаштування

In [1]:
# Спільне налаштування для всіх ноутбуків довідника.
# Шукає корінь репозиторію, щоб шляхи на кшталт research/... працювали
# незалежно від того, звідки запущено Jupyter.
import pathlib
import sys

_cwd = pathlib.Path.cwd()
if (_cwd / "research").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "research").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Не знайдено каталог research/ ні в {_cwd}, ні в {_cwd.parent}. "
        "Запускайте Jupyter з кореня репозиторію або з теки notebooks/."
    )

print("Python :", sys.version.split()[0])
print("Корінь :", ROOT)

Python : 3.12.3
Корінь : /home/volodymyr/PycharmProjects/ai-engineer-docs


In [2]:
# Версії бібліотек, використаних у цьому ноутбуку (для відтворюваності).
import importlib

for _name in ("nbformat", "jinja2", "transformers", "anthropic"):
    try:
        _mod = importlib.import_module(_name)
        print(f"{_name:14} {getattr(_mod, '__version__', '?')}")
    except ImportError:
        print(f"{_name:14} НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)")

nbformat       5.9.1
jinja2         3.1.2
transformers   НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)
anthropic      НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)


## 7.1 Формула візуальних токенів

Документація Anthropic формулює її так: зображення розглядається як **патчі 28×28 пікселя**, кожен
патч — один візуальний токен.

```
візуальні токени = ⌈ширина / 28⌉ × ⌈висота / 28⌉
```

Реалізуємо й **звіримо з таблицею документації** — це найкраща перевірка, що формулу зрозуміло
правильно.

In [3]:
import math


def visual_tokens(width: int, height: int) -> int:
    """Візуальні токени зображення: патчі 28x28 пікселя.

    Формула з документації: ceil(width / 28) * ceil(height / 28).
    Застосовна до зображень, які НЕ перевищують межі моделі.
    """
    return math.ceil(width / 28) * math.ceil(height / 28)


# ── Звірка з таблицею документації ───────────────────────────────────────
# Ці значення взяті з таблиці «downsized resolution and visual-token cost».
DOCUMENTED = [
    (200,   200,   64,   "без зменшення"),
    (1000,  1000,  1296, "без зменшення"),
    (1092,  1092,  1521, "без зменшення"),
    (1920,  1080,  2691, "висока роздільність, без зменшення"),
    (2000,  1500,  3888, "висока роздільність, без зменшення"),
]

print(f"{'розмір':>11} {'формула':>9} {'документація':>13}  {'збіг':>6}  примітка")
print("-" * 74)
all_ok = True
for w, h, expected, note in DOCUMENTED:
    got = visual_tokens(w, h)
    ok = got == expected
    all_ok &= ok
    print(f"{w}x{h:>5} {got:>9} {expected:>13}  {('OK' if ok else 'НІ'):>6}  {note}")

print()
print("Усі значення збіглися:", all_ok)
print()
print("Зверніть увагу на 1920x1080: 2691 — це ВИСОКОРОЗДІЛЬНИЙ рядок (без зменшення).")
print("На стандартному рівні те саме зображення зменшується до 1456x819 і коштує 1560.")

     розмір   формула  документація    збіг  примітка
--------------------------------------------------------------------------
200x  200        64            64      OK  без зменшення
1000x 1000      1296          1296      OK  без зменшення
1092x 1092      1521          1521      OK  без зменшення
1920x 1080      2691          2691      OK  висока роздільність, без зменшення
2000x 1500      3888          3888      OK  висока роздільність, без зменшення

Усі значення збіглися: True

Зверніть увагу на 1920x1080: 2691 — це ВИСОКОРОЗДІЛЬНИЙ рядок (без зменшення).
На стандартному рівні те саме зображення зменшується до 1456x819 і коштує 1560.


Формула **лінійна за площею**: подвоєння обох сторін учетверо збільшує кількість патчів. Саме тому
попереднє зменшення — такий дієвий важіль економії.

In [4]:
# Два рівні роздільності з документації
TIERS = {
    "стандартний":        {"max_long_edge": 1568, "max_visual_tokens": 1568,
                           "models": "усі, крім 4.7+"},
    "висока роздільність": {"max_long_edge": 2576, "max_visual_tokens": 4784,
                            "models": "Claude 4.7 і пізніші"},
}

print(f"{'рівень':22} {'довга сторона':>14} {'макс. токенів':>14}  моделі")
print("-" * 76)
for name, t in TIERS.items():
    print(f"{name:22} {t['max_long_edge']:>14} {t['max_visual_tokens']:>14}  {t['models']}")

print()
print("Підтримка високої роздільності АВТОМАТИЧНА на зазначених моделях —")
print("ні бета-заголовка, ні opt-in на боці клієнта не потрібно.")

print()
print("Наслідок для бюджету: те саме зображення на двох рівнях:")
w, h = 1920, 1080
print(f"  {w}x{h} на стандартному рівні: 1560 токенів (після зменшення)")
print(f"  {w}x{h} на високій роздільності: {visual_tokens(w,h)} токенів")
print(f"  різниця: {visual_tokens(w,h)/1560:.2f}×")

рівень                  довга сторона  макс. токенів  моделі
----------------------------------------------------------------------------
стандартний                      1568           1568  усі, крім 4.7+
висока роздільність              2576           4784  Claude 4.7 і пізніші

Підтримка високої роздільності АВТОМАТИЧНА на зазначених моделях —
ні бета-заголовка, ні opt-in на боці клієнта не потрібно.

Наслідок для бюджету: те саме зображення на двох рівнях:
  1920x1080 на стандартному рівні: 1560 токенів (після зменшення)
  1920x1080 на високій роздільності: 2691 токенів
  різниця: 1.73×


In [5]:
# Скільки коштує одне зображення? Формула + ціна за токен.
PRICE_PER_MTOK = 5.00   # Claude Opus 5, вхід

SAMPLES = [
    (200, 200, 64, "таблиця документації"),
    (512, 512, visual_tokens(512, 512), "формула (без зменшення)"),
    (1024, 768, visual_tokens(1024, 768), "формула (без зменшення)"),
    (1920, 1080, 2691, "таблиця документації (висока розд.)"),
    (3840, 2160, 4784, "таблиця документації (висока розд.)"),
]

print(f"{'зображення':>11} {'токенів':>9} {'ціна':>12}  джерело числа")
print("-" * 70)
for w, h, tokens, src in SAMPLES:
    cost = tokens / 1_000_000 * PRICE_PER_MTOK
    print(f"{w}x{h:>5} {tokens:>9} ${cost:>11.6f}  {src}")

print()
print("Для масштабу: одне зображення 1920x1080 ≈ 2700 токенів входу —")
print("це приблизно як сторінка щільного тексту (розділ 3).")
print()
print("Тому тисяча зображень на день на Opus 5 коштує:")
for tokens, label in ((1560, "стандартний рівень"), (2691, "висока роздільність")):
    daily = 1000 * tokens / 1e6 * PRICE_PER_MTOK
    print(f"  {label:22} ${daily:.2f}/день  (${daily*30:.0f}/місяць)")

 зображення   токенів         ціна  джерело числа
----------------------------------------------------------------------
200x  200        64 $   0.000320  таблиця документації
512x  512       361 $   0.001805  формула (без зменшення)
1024x  768      1036 $   0.005180  формула (без зменшення)
1920x 1080      2691 $   0.013455  таблиця документації (висока розд.)
3840x 2160      4784 $   0.023920  таблиця документації (висока розд.)

Для масштабу: одне зображення 1920x1080 ≈ 2700 токенів входу —
це приблизно як сторінка щільного тексту (розділ 3).

Тому тисяча зображень на день на Opus 5 коштує:
  стандартний рівень     $7.80/день  ($234/місяць)
  висока роздільність    $13.45/день  ($404/місяць)


## 7.1 Пастка багато-зображувальних запитів

Найпідступніше обмеження: якщо запит містить **більше 20 зображень**, до **кожного** зображення
застосовується суворіший ліміт розмірів. І в цей поріг входять **усі** блоки `image` — включно із
зображеннями з попередніх ходів і з тими, що всередині `tool_result`.

Відтворимо цю логіку.

In [6]:
MANY_IMAGE_THRESHOLD = 20

def image_limit_status(total_images_in_request: int) -> dict:
    """Визначає, який режим лімітів діє для запиту."""
    strict = total_images_in_request > MANY_IMAGE_THRESHOLD
    return {
        "зображень у запиті": total_images_in_request,
        "суворіший ліміт": strict,
        "режим": "MANY-IMAGE requests (суворіший)" if strict else "звичайний",
    }


print(f"{'зображень':>10} {'режим':>32}")
print("-" * 46)
for n in (1, 10, 20, 21, 50):
    r = image_limit_status(n)
    print(f"{n:>10} {r['режим']:>32}")

print()
print("Поріг спрацьовує на 21-му зображенні, не на 20-му.")
print()
print("ЧОМУ ЦЕ НЕСПОДІВАНА ПАСТКА: у поріг входять не лише зображення з ПОТОЧНОГО")
print("повідомлення. Документація прямо каже, що враховуються всі блоки image:")
print("  • зображення з попередніх ходів розмови, які ви надсилаєте ПОВТОРНО")
print("  • зображення ВСЕРЕДИНІ вмісту tool_result (наприклад, скріншоти)")
print("  • на Bedrock і Google Cloud — також блоки документів (PDF)")
print()
print("Тобто довга розмова зі скріншотами може раптово почати відхиляти зображення,")
print("які раніше проходили — хоча ви не змінювали розмір жодного з них.")

 зображень                            режим
----------------------------------------------
         1                        звичайний
        10                        звичайний
        20                        звичайний
        21  MANY-IMAGE requests (суворіший)
        50  MANY-IMAGE requests (суворіший)

Поріг спрацьовує на 21-му зображенні, не на 20-му.

ЧОМУ ЦЕ НЕСПОДІВАНА ПАСТКА: у поріг входять не лише зображення з ПОТОЧНОГО
повідомлення. Документація прямо каже, що враховуються всі блоки image:
  • зображення з попередніх ходів розмови, які ви надсилаєте ПОВТОРНО
  • зображення ВСЕРЕДИНІ вмісту tool_result (наприклад, скріншоти)
  • на Bedrock і Google Cloud — також блоки документів (PDF)

Тобто довга розмова зі скріншотами може раптово почати відхиляти зображення,
які раніше проходили — хоча ви не змінювали розмір жодного з них.


In [7]:
# Симуляція: розмова зі скріншотами накопичує зображення
print("Розмова, де кожен хід додає 3 скріншоти (і історія пересилається щоразу):")
print()
print(f"{'хід':>4} {'у запиті':>9} {'режим':>32}")
print("-" * 50)
for turn in range(1, 10):
    total = turn * 3     # історія + поточний хід
    r = image_limit_status(total)
    mark = "  ← перейшли на суворіший" if total == 21 else ""
    print(f"{turn:>4} {total:>9} {r['режим']:>32}{mark}")

print()
print("Практичний висновок: у довгих розмовах зі скріншотами треба або")
print("зменшувати зображення заздалегідь, або не пересилати старі зображення.")

Розмова, де кожен хід додає 3 скріншоти (і історія пересилається щоразу):

 хід  у запиті                            режим
--------------------------------------------------
   1         3                        звичайний
   2         6                        звичайний
   3         9                        звичайний
   4        12                        звичайний
   5        15                        звичайний
   6        18                        звичайний
   7        21  MANY-IMAGE requests (суворіший)  ← перейшли на суворіший
   8        24  MANY-IMAGE requests (суворіший)
   9        27  MANY-IMAGE requests (суворіший)

Практичний висновок: у довгих розмовах зі скріншотами треба або
зменшувати зображення заздалегідь, або не пересилати старі зображення.


## 7.2 PDF: дві складові вартості

Сервер конвертує **кожну сторінку в зображення** й паралельно витягує текст. Модель отримує **і те,
і те**. Тому ви платите двічі — і це головна причина, чому PDF дорогий.

In [8]:
# Складові вартості PDF (з документації):
#   текст: 1 500 - 3 000 токенів на сторінку (залежить від щільності)
#   зображення: ті самі розрахунки, що для зображень
TEXT_TOKENS_PER_PAGE = (1500, 3000)

# ПРИПУЩЕННЯ: сторінка PDF дає ~1036 візуальних токенів (еквівалент 1024x768).
# Документація НЕ вказує точний розмір зображення, у яке конвертується сторінка.
ASSUMED_VISUAL_TOKENS = visual_tokens(1024, 768)


def pdf_page_cost_usd(price_per_mtok: float = PRICE_PER_MTOK) -> tuple[float, float]:
    """Діапазон вартості однієї сторінки PDF: (мінімум, максимум)."""
    lo = (TEXT_TOKENS_PER_PAGE[0] + ASSUMED_VISUAL_TOKENS) / 1e6 * price_per_mtok
    hi = (TEXT_TOKENS_PER_PAGE[1] + ASSUMED_VISUAL_TOKENS) / 1e6 * price_per_mtok
    return lo, hi


lo, hi = pdf_page_cost_usd()
print(f"Припущення про візуальні токени сторінки: {ASSUMED_VISUAL_TOKENS}")
print(f"Текст на сторінку (з документації): {TEXT_TOKENS_PER_PAGE[0]}-{TEXT_TOKENS_PER_PAGE[1]}")
print()
print(f"Одна сторінка: ${lo:.5f} - ${hi:.5f}")
print()
print(f"{'сторінок':>9} {'від':>9} {'до':>9}")
print("-" * 30)
for pages in (1, 10, 100, 300, 600):
    print(f"{pages:>9} ${pages*lo:>8.2f} ${pages*hi:>8.2f}")

print()
print("УВАГА: це ОЦІНКА ПОРЯДКУ. Точний розмір зображення сторінки в джерелах")
print("не вказано. Для справжнього бюджету міряйте конкретні PDF через count_tokens.")
print()
print("Головний висновок від цього не змінюється: 300 сторінок = одиниці доларів.")

Припущення про візуальні токени сторінки: 1036
Текст на сторінку (з документації): 1500-3000

Одна сторінка: $0.01268 - $0.02018

 сторінок       від        до
------------------------------
        1 $    0.01 $    0.02
       10 $    0.13 $    0.20
      100 $    1.27 $    2.02
      300 $    3.80 $    6.05
      600 $    7.61 $   12.11

УВАГА: це ОЦІНКА ПОРЯДКУ. Точний розмір зображення сторінки в джерелах
не вказано. Для справжнього бюджету міряйте конкретні PDF через count_tokens.

Головний висновок від цього не змінюється: 300 сторінок = одиниці доларів.


In [9]:
# Обмеження PDF із документації
PDF_LIMITS = {
    "максимальний розмір запиту": "32 МБ (різниться за платформами)",
    "максимум сторінок": "600 (100, якщо контекстне вікно < 1M токенів)",
    "формат": "стандартний PDF без паролів і шифрування",
}

for k, v in PDF_LIMITS.items():
    print(f"  {k:32} {v}")

print()
print("Обидві межі стосуються ВСЬОГО корисного навантаження запиту —")
print("включно з іншим вмістом, надісланим поряд із PDF.")
print()
print("Порада з документації: для великих PDF завантажуйте їх через Files API")
print("й посилайтеся за file_id, щоб тримати розмір запиту малим (див. 7.4).")

  максимальний розмір запиту       32 МБ (різниться за платформами)
  максимум сторінок                600 (100, якщо контекстне вікно < 1M токенів)
  формат                           стандартний PDF без паролів і шифрування

Обидві межі стосуються ВСЬОГО корисного навантаження запиту —
включно з іншим вмістом, надісланим поряд із PDF.

Порада з документації: для великих PDF завантажуйте їх через Files API
й посилайтеся за file_id, щоб тримати розмір запиту малим (див. 7.4).


## 7.3 Цитати: індексація, яка легко збиває з пантелику

Три типи документів мають **різний формат цитат**. Зверніть увагу на базовий індекс.

In [10]:
# Типи документів для цитат (з документації)
CITATION_DOC_TYPES = {
    "plain text":    {"поділ": "за реченнями", "формат": "індекси символів", "база": 0},
    "pdf":           {"поділ": "за реченнями", "формат": "номери сторінок", "база": 1},
    "custom content": {"поділ": "налаштовується", "формат": "залежить від структури", "база": None},
}

print(f"{'тип':16} {'поділ':>18} {'формат':>24} {'база':>6}")
print("-" * 68)
for name, t in CITATION_DOC_TYPES.items():
    base = "—" if t["база"] is None else str(t["база"])
    print(f"{name:16} {t['поділ']:>18} {t['формат']:>24} {base:>6}")

print()
print("ПАСТКА: символи індексуються З 0, сторінки — З 1.")
print("Це класичне джерело зсуву 'на одиницю' при підсвічуванні цитат в UI.")


def highlight(text: str, start: int, end: int, base: int = 0) -> str:
    """Підсвічує фрагмент за цитатою. base=0 для символів, base=1 для сторінок."""
    s, e = start - base, end - base      # перевести у 0-базований зріз Python
    return text[:s] + "[" + text[s:e] + "]" + text[e:]


doc = "Перше речення. Друге речення. Третє речення."
print()
print("Документ:", repr(doc))
print()
print("Цитата з base=0 (символи, plain text):")
print("  start=16, end=30 ->", highlight(doc, 16, 30, base=0))
print()
print("Якщо помилково застосувати base=1 до тих самих індексів:")
print("  start=16, end=30 ->", highlight(doc, 16, 30, base=1))
print("  ↑ фрагмент зсунутий на один символ")

тип                           поділ                   формат   база
--------------------------------------------------------------------
plain text             за реченнями         індекси символів      0
pdf                    за реченнями          номери сторінок      1
custom content       налаштовується   залежить від структури      —

ПАСТКА: символи індексуються З 0, сторінки — З 1.
Це класичне джерело зсуву 'на одиницю' при підсвічуванні цитат в UI.

Документ: 'Перше речення. Друге речення. Третє речення.'

Цитата з base=0 (символи, plain text):
  start=16, end=30 -> Перше речення. Д[руге речення. ]Третє речення.

Якщо помилково застосувати base=1 до тих самих індексів:
  start=16, end=30 -> Перше речення. [Друге речення.] Третє речення.
  ↑ фрагмент зсунутий на один символ


In [11]:
# Що можна цитувати, а що ні (з документації)
CITABLE = {
    "source (вміст документа)": True,
    "title": False,
    "context": False,
}

print("Процитований вміст:")
for field, citable in CITABLE.items():
    print(f"  {field:28} {'ТАК, цитується' if citable else 'НІ, лише передається моделі'}")

print()
print("Практичний наслідок: title і context НЕ цитуються, тому:")
print("  • НЕ кладіть вміст, який треба цитувати, у context")
print("  • title обмежений у довжині -> для метаданих використовуйте context")
print("  • context добре підходить для серіалізованого JSON із метаданими")
print()
print("Приклад метаданих у context:")
print('  {"dept": "legal", "date": "2026-09-01", "rev": 3}')
print()
print("А ось чому цитати вигідні: cited_text НЕ зараховується до вихідних токенів")
print("і НЕ зараховується до вхідних при передаванні в наступних ходах.")
print("Ви платите лише за компактні індекси розташування.")

Процитований вміст:
  source (вміст документа)     ТАК, цитується
  title                        НІ, лише передається моделі
  context                      НІ, лише передається моделі

Практичний наслідок: title і context НЕ цитуються, тому:
  • НЕ кладіть вміст, який треба цитувати, у context
  • title обмежений у довжині -> для метаданих використовуйте context
  • context добре підходить для серіалізованого JSON із метаданими

Приклад метаданих у context:
  {"dept": "legal", "date": "2026-09-01", "rev": 3}

А ось чому цитати вигідні: cited_text НЕ зараховується до вихідних токенів
і НЕ зараховується до вхідних при передаванні в наступних ходах.
Ви платите лише за компактні індекси розташування.


## 7.4 Files API: `file_id` — це можливість, а не просто ідентифікатор

Документація прямо попереджає: оскільки файли обмежені робочим простором і **будь-який** запит у
тому самому просторі може на них посилатися, **ніколи не приймайте `file_id` із недовірених джерел**.

Відтворимо, що станеться, якщо це правило порушити.

In [12]:
# Імітація сховища файлів, обмеженого робочим простором
FILES_STORE = {}     # file_id -> {"owner": user, "name": ...}


def upload(file_id: str, owner: str, name: str) -> None:
    FILES_STORE[file_id] = {"owner": owner, "name": name}


# Два користувачі в ОДНОМУ робочому просторі
upload("file_alice_salary", "alice", "Зарплатна відомість.xlsx")
upload("file_bob_report",   "bob",   "Мій звіт.pdf")


def vulnerable_handler(current_user: str, file_id_from_client: str) -> str:
    """УРАЗЛИВО: приймає file_id із запиту клієнта."""
    if file_id_from_client not in FILES_STORE:
        return "файл не знайдено"
    meta = FILES_STORE[file_id_from_client]
    return f"модель прочитала '{meta['name']}' (власник: {meta['owner']})"


def safe_handler(current_user: str, file_id_from_client: str) -> str:
    """БЕЗПЕЧНО: підставляє лише file_id, що належить поточному користувачу."""
    meta = FILES_STORE.get(file_id_from_client)
    if meta is None or meta["owner"] != current_user:
        return "доступ заборонено"
    return f"модель прочитала '{meta['name']}'"


print("Боб надсилає file_id, який належить Алісі:")
print()
print("  УРАЗЛИВИЙ обробник:")
print("   ", vulnerable_handler("bob", "file_alice_salary"))
print()
print("  БЕЗПЕЧНИЙ обробник:")
print("   ", safe_handler("bob", "file_alice_salary"))
print()
print("  Боб зі своїм власним файлом (безпечний обробник):")
print("   ", safe_handler("bob", "file_bob_report"))
print()
print("Це не ескалація привілеїв у класичному сенсі — це прямий обхід")
print("контролю доступу в межах робочого простору.")
print()
print("ПРАВИЛО: зберігайте відповідність 'користувач -> його file_id' у своїй базі")
print("й підставляйте ID лише звідти. Ніколи не беріть file_id із тіла запиту.")

Боб надсилає file_id, який належить Алісі:

  УРАЗЛИВИЙ обробник:
    модель прочитала 'Зарплатна відомість.xlsx' (власник: alice)

  БЕЗПЕЧНИЙ обробник:
    доступ заборонено

  Боб зі своїм власним файлом (безпечний обробник):
    модель прочитала 'Мій звіт.pdf'

Це не ескалація привілеїв у класичному сенсі — це прямий обхід
контролю доступу в межах робочого простору.

ПРАВИЛО: зберігайте відповідність 'користувач -> його file_id' у своїй базі
й підставляйте ID лише звідти. Ніколи не беріть file_id із тіла запиту.


In [13]:
# Життєвий цикл і обмеження Files API
LIFECYCLE = [
    ("Обмеження простором", "Будь-який запит у тому самому просторі може посилатися"),
    ("Недовірені джерела", "НІКОЛИ не приймайте file_id із недовірених джерел"),
    ("Зміна вмісту", "Неможлива: завантажте новий файл і видаліть старий"),
    ("Термін життя", "Доки не видалите через DELETE /v1/files/{file_id} або не настане expires_at"),
    ("Відновлення", "Видалені файли відновити НЕМОЖЛИВО"),
    ("Активні запити", "Одразу після видалення файли недоступні через API, але можуть залишатися в активних викликах"),
]

print(f"{'властивість':22} наслідок")
print("-" * 96)
for prop, effect in LIFECYCLE:
    print(f"{prop:22} {effect}")

print()
print("Обмеження сховища (з документації):")
print("  максимальний розмір файлу: 500 МБ")
print("  загальне сховище:          1 ТБ на організацію")
print()
print("Наслідок: якщо не керувати терміном життя, файли накопичуються до ліміту.")

властивість            наслідок
------------------------------------------------------------------------------------------------
Обмеження простором    Будь-який запит у тому самому просторі може посилатися
Недовірені джерела     НІКОЛИ не приймайте file_id із недовірених джерел
Зміна вмісту           Неможлива: завантажте новий файл і видаліть старий
Термін життя           Доки не видалите через DELETE /v1/files/{file_id} або не настане expires_at
Відновлення            Видалені файли відновити НЕМОЖЛИВО
Активні запити         Одразу після видалення файли недоступні через API, але можуть залишатися в активних викликах

Обмеження сховища (з документації):
  максимальний розмір файлу: 500 МБ
  загальне сховище:          1 ТБ на організацію

Наслідок: якщо не керувати терміном життя, файли накопичуються до ліміту.


## 7.5 Вирішувач: пряма передача чи попередня обробка

Пряма передача найпростіша в реалізації й найдорожча в експлуатації. Побудуємо простий вирішувач,
який радить підхід за властивостями задачі.

In [14]:
def recommend_approach(
    *,
    needs_visual_understanding: bool,
    needs_text_only: bool,
    page_count: int,
    questions_per_document: int,
    documents_per_day: int,
) -> dict:
    """Радить підхід до обробки документів.

    Це евристика на основі механіки з документації, а не формула з неї:
    числа порогів обрані для ілюстрації.
    """
    reasons = []

    if needs_visual_understanding:
        reasons.append("потрібне розуміння зображень — OCR не допоможе, "
                       "передавайте сторінки моделі")
        base = "пряма передача (з кешем)"
    elif page_count > 20 and needs_text_only:
        reasons.append(f"{page_count} сторінок і потрібен лише текст — "
                       "витягніть текст замість зображень")
        base = "витягування тексту"
    elif page_count <= 5:
        reasons.append("документ малий — попередня обробка не окупиться")
        base = "пряма передача"
    else:
        base = "витягування тексту + кеш"

    if questions_per_document > 3:
        reasons.append(f"{questions_per_document} питань на документ — "
                       "увімкніть кешування промпту")
    if documents_per_day > 500:
        reasons.append(f"{documents_per_day} документів на день — "
                       "розгляньте батч-обробку (знижка 50%) і RAG")
    if page_count > 100:
        reasons.append(f"{page_count} сторінок — відбирайте потрібні сторінки, "
                       "не надсилайте весь документ")

    return {"рекомендація": base, "обґрунтування": reasons}


SCENARIOS = {
    "Аналіз графіків у звіті (3 стор.)": dict(
        needs_visual_understanding=True, needs_text_only=False,
        page_count=3, questions_per_document=1, documents_per_day=10),
    "Договір 250 стор., 1 питання": dict(
        needs_visual_understanding=False, needs_text_only=True,
        page_count=250, questions_per_document=1, documents_per_day=5),
    "Той самий PDF, 10 питань": dict(
        needs_visual_understanding=False, needs_text_only=True,
        page_count=40, questions_per_document=10, documents_per_day=2),
    "1000 рахунків на день": dict(
        needs_visual_understanding=False, needs_text_only=True,
        page_count=2, questions_per_document=1, documents_per_day=1000),
}

for label, params in SCENARIOS.items():
    r = recommend_approach(**params)
    print(f"--- {label} ---")
    print(f"    рекомендація: {r['рекомендація']}")
    for reason in r["обґрунтування"]:
        print(f"      • {reason}")
    print()

--- Аналіз графіків у звіті (3 стор.) ---
    рекомендація: пряма передача (з кешем)
      • потрібне розуміння зображень — OCR не допоможе, передавайте сторінки моделі

--- Договір 250 стор., 1 питання ---
    рекомендація: витягування тексту
      • 250 сторінок і потрібен лише текст — витягніть текст замість зображень
      • 250 сторінок — відбирайте потрібні сторінки, не надсилайте весь документ

--- Той самий PDF, 10 питань ---
    рекомендація: витягування тексту
      • 40 сторінок і потрібен лише текст — витягніть текст замість зображень
      • 10 питань на документ — увімкніть кешування промпту

--- 1000 рахунків на день ---
    рекомендація: пряма передача
      • документ малий — попередня обробка не окупиться
      • 1000 документів на день — розгляньте батч-обробку (знижка 50%) і RAG



## Самоперевірка

Перевіряємо формулу проти документації, арифметику цін, пороги й логіку безпеки.

In [15]:
# ── 1. Формула збігається з таблицею документації ───────────────────────
assert visual_tokens(200, 200) == 64
assert visual_tokens(1000, 1000) == 1296
assert visual_tokens(1092, 1092) == 1521
assert visual_tokens(1920, 1080) == 2691     # високороздільний рядок
assert visual_tokens(2000, 1500) == 3888
assert all_ok, "усі рядки таблиці мусять збігатися"
print("✓ формула візуальних токенів збігається з таблицею документації (5/5)")

# ── 2. Формула лінійна за площею: 2x сторона = 4x токенів ───────────────
assert visual_tokens(56, 56) == 4            # 2x2 патчі
assert visual_tokens(112, 112) == 16         # 4x4 патчі
assert visual_tokens(112, 112) == 4 * visual_tokens(56, 56)
print("✓ подвоєння сторін збільшує токени вчетверо")

# ── 3. Рівні роздільності узгоджені з джерелом ──────────────────────────
assert TIERS["стандартний"]["max_long_edge"] == 1568
assert TIERS["стандартний"]["max_visual_tokens"] == 1568
assert TIERS["висока роздільність"]["max_long_edge"] == 2576
assert TIERS["висока роздільність"]["max_visual_tokens"] == 4784
print("✓ рівні роздільності: 1568/1568 і 2576/4784")

# ── 4. Поріг багато-зображувальних запитів спрацьовує на 21 ─────────────
assert image_limit_status(20)["суворіший ліміт"] is False
assert image_limit_status(21)["суворіший ліміт"] is True
print("✓ поріг many-image: 20 зображень — звичайний режим, 21 — суворіший")

# ── 5. Ціна зображення лінійна за токенами ──────────────────────────────
c_1k = 1036 / 1e6 * PRICE_PER_MTOK
c_2k = 2691 / 1e6 * PRICE_PER_MTOK
assert c_2k > c_1k
assert abs(c_1k * 2 - 1036 * 2 / 1e6 * PRICE_PER_MTOK) < 1e-12
print(f"✓ ціна: 1036 токенів = ${c_1k:.6f}, 2691 токенів = ${c_2k:.6f}")

# ── 6. Вартість PDF зростає лінійно за сторінками ───────────────────────
lo1, hi1 = pdf_page_cost_usd()
assert hi1 > lo1, "верхня межа мусить бути більшою за нижню"
assert abs(hi1 * 300 / (hi1 * 100) - 3) < 1e-9
print(f"✓ PDF: сторінка ${lo1:.5f}-${hi1:.5f}, 300 сторінок ${lo1*300:.2f}-${hi1*300:.2f}")

# ── 7. Обмеження PDF узгоджені з джерелом ───────────────────────────────
assert "32 МБ" in PDF_LIMITS["максимальний розмір запиту"]
assert "600" in PDF_LIMITS["максимум сторінок"]
print("✓ обмеження PDF: 32 МБ на запит, 600 сторінок")

# ── 8. Індексація цитат: символи з 0, сторінки з 1 ──────────────────────
assert CITATION_DOC_TYPES["plain text"]["база"] == 0
assert CITATION_DOC_TYPES["pdf"]["база"] == 1
# Демонстрація, що помилкова база дає інший фрагмент
assert highlight(doc, 16, 30, base=0) != highlight(doc, 16, 30, base=1)
print("✓ індексація цитат: символи з 0, сторінки з 1 (помилка зсуває фрагмент)")

# ── 9. Цитуються лише джерела, не title і не context ────────────────────
assert CITABLE["source (вміст документа)"] is True
assert CITABLE["title"] is False
assert CITABLE["context"] is False
print("✓ цитати: source цитується, title і context — ні")

# ── 10. Уразливий обробник читає чужий файл, безпечний — ні ─────────────
leaked = vulnerable_handler("bob", "file_alice_salary")
assert "alice" in leaked, "уразливий обробник мусить прочитати чужий файл"
blocked = safe_handler("bob", "file_alice_salary")
assert blocked == "доступ заборонено"
assert safe_handler("bob", "file_bob_report") != "доступ заборонено"
print("✓ Files API: уразливий обробник читає чужий файл, безпечний блокує")

# ── 11. Обмеження сховища узгоджені з джерелом ──────────────────────────
assert len(LIFECYCLE) == 6
print("✓ життєвий цикл Files API: 6 властивостей задокументовано")

# ── 12. Вирішувач радить різні підходи для різних задач ─────────────────
r_visual = recommend_approach(needs_visual_understanding=True, needs_text_only=False,
                              page_count=3, questions_per_document=1, documents_per_day=10)
r_mass = recommend_approach(needs_visual_understanding=False, needs_text_only=True,
                            page_count=2, questions_per_document=1, documents_per_day=1000)
assert "пряма передача" in r_visual["рекомендація"]
assert any("батч" in x for x in r_mass["обґрунтування"])
print("✓ вирішувач: різні задачі -> різні рекомендації")

print()
print("Усі перевірки пройдено.")

✓ формула візуальних токенів збігається з таблицею документації (5/5)
✓ подвоєння сторін збільшує токени вчетверо
✓ рівні роздільності: 1568/1568 і 2576/4784
✓ поріг many-image: 20 зображень — звичайний режим, 21 — суворіший
✓ ціна: 1036 токенів = $0.005180, 2691 токенів = $0.013455
✓ PDF: сторінка $0.01268-$0.02018, 300 сторінок $3.80-$6.05
✓ обмеження PDF: 32 МБ на запит, 600 сторінок
✓ індексація цитат: символи з 0, сторінки з 1 (помилка зсуває фрагмент)
✓ цитати: source цитується, title і context — ні
✓ Files API: уразливий обробник читає чужий файл, безпечний блокує
✓ життєвий цикл Files API: 6 властивостей задокументовано
✓ вирішувач: різні задачі -> різні рекомендації

Усі перевірки пройдено.


## Підсумок

**Головне з цього ноутбука:**

1. **Модель бачить патчі, не пікселі.** Токени = `⌈w/28⌉ × ⌈h/28⌉`. Формула збігається з таблицею
   документації для всіх п'яти перевірених розмірів.
2. **Два рівні роздільності:** стандартний (1568 px / 1568 токенів) і високий
   (Claude 4.7+: 2576 px / 4784 токени). Високий — автоматичний, без бета-заголовка.
3. **Велике зображення на стандартному рівні не має сенсу:** 1920×1080, 2000×1500 і 3840×2160
   коштують майже однаково (1560 / 1564 / 1560), бо всі зменшуються до однієї межі.
4. **Пастка багато-зображувальних запитів:** понад 20 зображень у запиті вмикає **суворіший ліміт
   розмірів для кожного** зображення. У поріг входять зображення з попередніх ходів і з `tool_result`.
5. **PDF коштує двічі:** 1 500–3 000 токенів тексту на сторінку **плюс** візуальні токени за
   зображення сторінки. 300 сторінок — одиниці доларів лише за читання.
6. **Виняток для computer/browser use:** їхні зображення **не зменшуються** — API їх **відхиляє**.
   Зменшуйте у своєму застосунку.
7. **Цитати індексуються по-різному:** символи з 0, сторінки з 1. Цитується лише вміст `source`;
   `title` і `context` лише передаються моделі.
8. **`cited_text` не коштує** ні вихідних токенів, ні вхідних при передаванні далі — цитати
   ефективніші, ніж здається.
9. **`file_id` — це можливість.** Ніколи не приймайте його від клієнта: файли обмежені робочим
   простором, і чужий ID дає доступ до чужого файлу.
10. **Пряма передача ≠ безкоштовна простота.** Для великого обсягу витягування тексту, відбір
    сторінок, кеш і батч зменшують рахунок у рази.

**Куди далі:**

- Розділ 3 — підрахунок токенів до відправки, включно із зображеннями (потрібен base64).
- Розділ 9 — кешування документів і зображень: найбільший важіль для повторного аналізу.
- Розділ 10 — батч-обробка: знижка 50% для масової обробки документів.
- Розділ 13 — computer use і модель загроз для скриншотів.
- Розділ 16 — RAG: індексація замість передавання документів.

## Джерела

- [Anthropic — Vision](https://platform.claude.com/docs/en/build-with-claude/vision)
- [Anthropic — PDF support](https://platform.claude.com/docs/en/build-with-claude/pdf-support)
- [Anthropic — Citations](https://platform.claude.com/docs/en/build-with-claude/citations)
- [Anthropic — Files API](https://platform.claude.com/docs/en/build-with-claude/files)

Джерела збережено локально в `research/02/`: `vision.md`, `pdf.md`, `citations.md`, `files-api.md`.